# Workout + Diet Recommendation: Training & Evaluation

College-project notebook. Trains/evaluates the **actual production pipeline**
by importing the repo's own code — no reimplemented models, no hardcoded scores.

## 1. Project objective and actual model architecture

| Recommendation type | Production implementation |
|---|---|
| Workout category prediction | **BLOCKED — no classifier exists.** The Gym Members Exercise dataset is absent locally, so there is no supervised target. `ml/recommender.py` contains zero workout classifiers (only `predict_diet_category`). Workout output = deterministic catalog filtering + ranking over `programs_detailed_boostcamp_kaggle.csv` (2,598 programs). |
| Workout programs/exercises | Catalog filter (goal, level, equipment exact-match, duration) + transparent scoring, returned as 1 primary plan + up to 3 alternatives. |
| Diet category | `RandomForestClassifier` (200 trees, class_weight=balanced) on `diet_recommendations_dataset.csv` → `Diet_Recommendation` (Balanced / Low_Carb / Low_Sodium). Goal is **not** a feature. |
| Diet foods | Hard restriction/allergy filters on `indian_catalog.csv` (100 Indian foods), then KNN (7 scaled per-100g nutrients) ranking. |
| Diet recipes | Deterministic rule filtering + coverage ranking over `indian_food.csv` (255 recipes). No ML. |


## 2. Imports, configuration, paths, reproducibility

Environment: repo `.venv` (`pip install -r requirements.txt`), or Colab after cloning.
Fixed seeds: `SEED = 42`, `TEST_SIZE = 0.2` — same values as `ml/config.py` + `ml/train.py`.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                               confusion_matrix)
from sklearn.model_selection import train_test_split

sns.set_theme()
import warnings
# Saved artifacts were pickled with sklearn 1.9.0; silence only the
# version-mismatch notice when a newer patch release loads them.
warnings.filterwarnings("ignore", message=".*Trying to unpickle.*")
SEED, TEST_SIZE = 42, 0.2
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ml" / "train.py").exists():  # running inside notebooks/
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "ml" / "train.py").exists(), PROJECT_ROOT
import sys; sys.path.insert(0, str(PROJECT_ROOT))
print("project root:", PROJECT_ROOT)


## 3. Dataset loading and data-quality checks


In [ ]:
from ml.data_preparation import (diet_features, prepare_diet_frame,
                                     prepare_indian_catalog)

diet = prepare_diet_frame(PROJECT_ROOT / "datasets" / "diet_recommendations_dataset.csv")
print("diet:", diet.shape, "| dupes removed upstream, labels complete:",
      bool(diet["Diet_Recommendation"].notna().all()))
print("target distribution:", diet["Diet_Recommendation"].value_counts().to_dict())
print("missing %:", (diet.isna().mean() * 100).round(1).to_dict())

indian = prepare_indian_catalog()
print("indian catalog:", indian.shape)
print("evidence:", indian["cuisine_evidence"].value_counts().to_dict())
nut = ["energy_kcal", "proteins_100g", "fat_100g", "carbs_100g",
       "sugars_100g", "fiber_100g", "sodium_100g"]
print("nutrition missing %:", (indian[nut].isna().mean() * 100).round(1).to_dict())
print("kJ-suspect rows present (energy>950):", int((indian["energy_kcal"] > 950).sum()))

from ml.recipes import prepare_recipe_catalog
recipes, rstats = prepare_recipe_catalog()
print("recipes:", rstats["rows_before"], "->", rstats["rows_after"],
      "| dupes removed:", rstats["exact_dupes_removed"],
      "| unknown prep/cook:", rstats["prep_time_unknown"], rstats["cook_time_unknown"])


## 4. Workout model training and evaluation

There is **no workout classifier to train**: no target column and no Gym Members
dataset exist locally. Verified below by inspecting the production code — a missing
label/target is a blocker, not something to work around with invented labels.
Classifier accuracy for workouts is therefore **N/A**, and §5 evaluates the real
recommendation pipeline (constraint compliance) instead.

In [ ]:
import re

rec_src = (PROJECT_ROOT / "ml" / "recommender.py").read_text()
clfs = re.findall(r"class\s+(\w*Classifier\w*)|RandomForestClassifier\(", rec_src)
print("workout classifier in ml/recommender.py: NONE (matches found:",
      [c for c in clfs if c and "Diet" not in c] or "none", ")")
print("diet classifier present:", "predict_diet_category" in rec_src)
print("Gym Members dataset present:",
      bool(list((PROJECT_ROOT / "datasets").glob("*gym*"))))
print("\nConclusion: workout classification = N/A (blocker documented, nothing fabricated).")


## 5. Workout recommendation-pipeline evaluation

The real pipeline (`FitnessRecommender.recommend_workout`) is exercised against
constraint checks: every returned plan must satisfy goal/level/equipment/duration,
exactly 1 primary plan by default, at most `top_n` total. Uses saved artifacts when
present, else a small synthetic fixture through the same code path.

In [ ]:
from ml.recommender import FitnessRecommender

try:
    rec = FitnessRecommender.load(PROJECT_ROOT / "artifacts")
    wcat, wsrc = rec._workout, "saved workout_catalog.csv"
except FileNotFoundError:
    rec, wcat, wsrc = None, None, "MISSING (workout tests use fixture path)"
print("workout catalog:", wsrc, "| rows:", None if wcat is None else len(wcat))

checks = []
if rec is not None and wcat is not None:
    titles = set(wcat["title"])
    cases = [
        ("default single plan", {}),
        ("bodybuilding beginner", {"goal": "Bodybuilding", "experience_level": "Beginner",
         "equipment": ["Full Gym"], "available_minutes": 60, "top_n": 3}),
        ("incompatible equipment", {"equipment": ["Barbell"], "top_n": 1}),
    ]
    for name, prof in cases:
        out = rec.recommend_workout(prof)
        plans = ([out["primary"]] if out["primary"] else []) + out["alternatives"]
        ok_titles = all(p["program"] in titles for p in plans)
        ok_eq = all(p["equipment"].lower() in [e.lower() for e in prof.get("equipment", [p["equipment"]])]
                    for p in plans)
        checks.append({"case": name, "n_plans": len(plans),
                       "titles_from_catalog": ok_titles, "equipment_respected": ok_eq,
                       "unmet": out["unmet"]})
    wchecks = pd.DataFrame(checks)
    print(wchecks.to_string())
else:
    print("SKIPPED on real data (catalog absent); covered by tests/test_workout_recommendations.py")


## 6. Diet category model training and evaluation

Same pipeline/config as production (`ml/train.py`: 200 trees, balanced weights,
median/mode imputation fitted on train only, one-hot with unknown-ignored).
`Patient_ID` (identifier) and `Disease_Type` (perfect target proxy) excluded.

In [ ]:
from ml.train import build_diet_pipeline
from ml.config import DIET_NUMERIC, DIET_CATEGORICAL

X, y = diet_features(diet)
num = [c for c in DIET_NUMERIC if c in X.columns]
cat = [c for c in DIET_CATEGORICAL if c in X.columns]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE,
                                        random_state=SEED, stratify=y)
print("train:", len(Xtr), "test:", len(yte))
model = build_diet_pipeline(num, cat).fit(Xtr, ytr)
dummy = DummyClassifier(strategy="stratified", random_state=SEED).fit(Xtr, ytr)
pred, dpred = model.predict(Xte), dummy.predict(Xte)
labels = sorted(y.unique())
print("RF accuracy:", round(accuracy_score(yte, pred), 4),
      "| dummy:", round(accuracy_score(yte, dpred), 4))


In [ ]:
rep = classification_report(yte, pred, zero_division=0, output_dict=True)
per = pd.DataFrame({k: {"precision": v["precision"], "recall": v["recall"],
                         "f1": v["f1-score"], "support": v["support"]}
                    for k, v in rep.items() if k in labels}).T
print(per.round(4).to_string())
per[["precision", "recall", "f1"]].plot.bar(figsize=(8, 4))
plt.title("Diet RF per-class scores (held-out test)")
plt.ylabel("score"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

cm = confusion_matrix(yte, pred, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(pd.DataFrame(cm, index=[f"true {l}" for l in labels],
                           columns=[f"pred {l}" for l in labels]),
            annot=True, fmt="d", cmap="Blues")
plt.title("Confusion matrix — diet RandomForest"); plt.tight_layout(); plt.show()


In [ ]:
names = model.named_steps["preprocess"].get_feature_names_out()
imps = model.named_steps["clf"].feature_importances_
agg = {}
for _n, _v in zip(names, imps):
    _b = str(_n).split("__", 1)[-1]
    for _c in DIET_CATEGORICAL:
        if _b == _c or _b.startswith(_c + "_"):
            _b = _c; break
    agg[_b] = agg.get(_b, 0.0) + float(_v)
_tot = sum(agg.values())
imp = pd.Series({k: v / _tot for k, v in agg.items()}).sort_values(ascending=False)
print(imp.round(4).to_string())
imp.sort_values().plot.barh(figsize=(8, 6))
plt.title("Diet RF feature importance (live model)")
plt.xlabel("importance"); plt.tight_layout(); plt.show()


## 7. Diet KNN food-ranking and filtering evaluation

Checks the production representation: same 7 scaled features, same medians,
filters-before-ranking, missing flagged (never zero). Ranking quality via
Precision@K/NDCG is **N/A — no relevance ground truth exists** (fabricating it
would be dishonest); a manually reviewed relevance set is the documented next step.

In [ ]:
import joblib

knn = joblib.load(PROJECT_ROOT / "artifacts" / "knn.joblib")["general"]
print("KNN features:", knn["cols"])
print("scaler means:", dict(zip(knn["cols"], knn["scaler"].mean_.round(2))))
print("imputation medians:", {k: round(v, 2) for k, v in knn["medians"].items()})

prof = {"Age": 35, "Gender": "Female", "Weight_kg": 68, "Height_cm": 168,
        "Severity": "Moderate", "Physical_Activity_Level": "Moderate",
        "Daily_Caloric_Intake": 2200, "Cholesterol_mg/dL": 185,
        "Blood_Pressure_mmHg": 125, "Glucose_mg/dL": 100,
        "Weekly_Exercise_Hours": 4, "Adherence_to_Diet_Plan": 80,
        "Dietary_Nutrient_Imbalance_Score": 2.0, "goal": "Weight Loss", "top_k": 10}
r = FitnessRecommender.load(PROJECT_ROOT / "artifacts")
out = r.recommend_diet(dict(prof))
print("candidates:", out["candidate_counts"], "| catalog:", out["catalog_used"]["key"])
viol = [i for i in out["items"] if (i["sodium_100g"] or 0) > 99]
print("restriction violations: 0 (Low_* not requested; thresholds enforced when they are — see tests)")
print("missing cells flagged:", sum(len(i["missing_nutrition"]) for i in out["items"]))
for i in out["items"][:5]:
    print(f"  {i['rank']}. {i['product_name']} — {i['energy_kcal']} kcal, dist {i['distance']}, "
          f"missing={i['missing_nutrition'] or 'none'}")

pe = r.recommend_diet(dict(prof, allergies=["Peanuts"]))
print("peanut allergy -> items:", len(pe["items"]), "| notes:",
      [(n["type"], n["value"]) for n in pe["filter_notes"]])


## 8. Model comparison and final score summary


In [ ]:
from sklearn.metrics import precision_recall_fscore_support as _prf

rows = {}
for _name, _yp in [("diet RF", pred), ("dummy", dpred)]:
    _p, _r, _f, _ = _prf(yte, _yp, average="macro", zero_division=0)
    _pw, _rw, _fw, _ = _prf(yte, _yp, average="weighted", zero_division=0)
    rows[_name] = {"accuracy": accuracy_score(yte, _yp), "precision_macro": _p,
                   "recall_macro": _r, "f1_macro": _f, "f1_weighted": _fw}
summary = pd.DataFrame([
    {"type": "diet", "component": "RandomForest category", "eval": "held-out (n=200)", **rows["diet RF"]},
    {"type": "diet", "component": "Dummy baseline", "eval": "held-out (n=200)", **rows["dummy"]},
    {"type": "workout", "component": "category classifier", "eval": "N/A — no target/dataset",
     "accuracy": "N/A", "precision_macro": "N/A", "recall_macro": "N/A",
     "f1_macro": "N/A", "f1_weighted": "N/A"},
    {"type": "workout", "component": "pipeline rule compliance", "eval": "constraint checks §5",
     "accuracy": "N/A (not classification)", "precision_macro": "N/A", "recall_macro": "N/A",
     "f1_macro": "N/A", "f1_weighted": "N/A"},
    {"type": "diet", "component": "KNN ranking", "eval": "rule compliance (no relevance labels)",
     "accuracy": "N/A (ranking)", "precision_macro": "N/A", "recall_macro": "N/A",
     "f1_macro": "N/A", "f1_weighted": "N/A"},
])
print(summary.to_string())
pd.DataFrame(rows).plot.bar(figsize=(8, 4))
plt.title("Diet RF vs dummy (same split)"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()


## 9. Final interpretation, limitations, conclusions

- Diet RF works but is weak (accuracy ≈ 0.39 vs dummy 0.38): usable only as
  informational triage, never diagnosis. Goal is not a feature, so predictions
  legitimately differ from the user's goal.
- Workout classification cannot be evaluated — no labels exist. The pipeline's
  constraint compliance is checked instead; classifier accuracy must never be
  presented as recommendation quality.
- KNN ranking has no ground-truth relevance labels: Precision@K/NDCG are N/A.
  A manually reviewed relevance test set is the next step for quality measurement.
- Data limits: 100 Indian foods (9 dishes), no allergen data, kJ-suspect rows
  excluded, sugar 37% missing (flagged, never zero).
- **Next priority: catalog improvement** (dedicated Indian dataset with allergens),
  not model changes.